A) Coverage: did every .pt get upgraded?

In [61]:
from pathlib import Path

def coverage(split):
    base = Path(f"Dataset/{split}")
    src = base/"hetero_ready"
    dst = base/"hetero_ready_gcbert"
    n_src = len(list(src.glob("*.pt")))
    n_dst = len(list(dst.glob("*.pt")))
    print(f"[{split}] src={n_src}  upgraded={n_dst}  missing={n_src-n_dst}")

for s in ("train","valid","test"):
    coverage(s)

[train] src=3438  upgraded=3436  missing=2
[valid] src=2905  upgraded=651  missing=2254
[test] src=2915  upgraded=0  missing=2915


B) Embedding health: shapes & norms look sane?

In [62]:
import torch, numpy as np
from pathlib import Path

def embedding_stats(split, sample=10):
    outs = sorted(Path(f"Dataset/{split}/hetero_ready_gcbert").glob("*.pt"))[:sample]
    norms = []
    for p in outs:
        d = torch.load(p, map_location="cpu")
        nt = getattr(d, "node_types", ())
        node_type = "node" if "node" in nt else (nt[0] if nt else next(iter(d.node_stores))._key)
        x = getattr(d[node_type], "x_text", None)
        assert x is not None, f"{p.name} has no x_text"
        assert x.ndim==2 and x.size(1)==768, f"{p.name} wrong shape: {tuple(x.shape)}"
        norms.append(x.norm(dim=1).numpy())
    arr = np.concatenate(norms)
    print(f"[{split}] x_text L2-norm: mean={arr.mean():.3f} std={arr.std():.3f} min={arr.min():.3f} max={arr.max():.3f} n={arr.size}")

embedding_stats("train")


[train] x_text L2-norm: mean=10.308 std=0.957 min=8.694 max=15.576 n=60854


In [20]:
import json, torch, random
from pathlib import Path

def peek(split):
    out_dir = Path(f"Dataset/{split}/hetero_ready_gcbert")
    json_dir= Path(f"Dataset/{split}/unified_aug")
    p = random.choice(sorted(out_dir.glob("*.pt")))
    base = p.stem
    # find json
    j = None
    for suf in (".json",".aug.json",".unified.json",".jsonl"):
        cand = json_dir/f"{base}{suf}"
        if cand.exists(): j=cand; break
    d = torch.load(p, map_location="cpu")
    nt = getattr(d,"node_types",())
    node_type = "node" if "node" in nt else (nt[0] if nt else next(iter(d.node_stores))._key)
    x = d[node_type].x_text
    print("file:", p.name, "node_type:", node_type, "x_text:", tuple(x.shape))
    if j:
        nodes = json.loads(j.read_text("utf-8")).get("nodes", [])
        for i in range(min(2, len(nodes))):
            print(" • text:", (nodes[i].get("code") or nodes[i].get("name") or "")[:90].replace("\n"," "))
            print("   emb norm:", float(x[i].norm()))
peek("train")


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_6312\2750428841.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(p, map_location="cpu")


file: shard_7a7b.pt node_type: node x_text: (3632, 768)
 • text: cp2112_gpio_direction_input
   emb norm: 9.726960182189941
 • text: cp2112_read_req
   emb norm: 9.466747283935547


Check homogeneous edges and x_text

In [21]:
import torch, math, numpy as np, json
from pathlib import Path

def pick_node_type(data, prefer="node"):
    nts = tuple(getattr(data, "node_types", ()))
    if prefer in nts: return prefer
    return nts[0] if nts else next(iter(data.node_stores))._key

def check_files(split, bases, prefer="node"):
    src_dir = Path(f"Dataset/{split}/hetero_ready")
    out_dir = Path(f"Dataset/{split}/hetero_ready_gcbert")
    ok = 0
    norms = []
    for base in bases:
        src = src_dir / f"{base}.pt"
        out = out_dir / f"{base}.pt"
        if not out.exists():
            print(f"[MISS] {out.name} not found yet"); continue

        g_src = torch.load(src, map_location="cpu")
        g_out = torch.load(out, map_location="cpu")

        nt = pick_node_type(g_out, prefer)
        store = g_out[nt]

        # 1) x_text presence & shape
        if not hasattr(store, "x_text"):
            print(f"[BAD] {out.name}: missing x_text"); continue
        x = store.x_text
        if x.ndim != 2 or x.size(1) != 768:
            print(f"[BAD] {out.name}: x_text shape {tuple(x.shape)}"); continue
        if x.dtype != torch.float32:
            print(f"[WARN] {out.name}: x_text dtype {x.dtype} (expected float32)")

        # 2) values finite / non-zero
        if not torch.isfinite(x).all():
            print(f"[BAD] {out.name}: non-finite values"); continue
        l2 = x.norm(dim=1)
        norms.append(l2.numpy())
        if float((l2==0).float().mean()) > 0.001:
            print(f"[WARN] {out.name}: >0.1% zero vectors")

        # 3) node count match (same store in source)
        nt_src = pick_node_type(g_src, prefer)
        if g_src[nt_src].num_nodes != store.num_nodes:
            print(f"[BAD] {out.name}: node count mismatch src={g_src[nt_src].num_nodes} out={store.num_nodes}")
            continue

        # 4) edges still present
        # try a common homogeneous key; adapt if your edges are hetero
        ei = None
        try:
            ei = g_out[(nt, "to", nt)].edge_index
        except Exception:
            # fall back to any edge_index on the store
            ei = getattr(store, "edge_index", None)
        if ei is None or ei.numel()==0:
            print(f"[WARN] {out.name}: no edge_index found on {nt} (check your edge keys)")

        ok += 1

    if norms:
        arr = np.concatenate(norms)
        print(f"[OK] {ok}/{len(bases)} passed. Norms: mean={arr.mean():.3f} std={arr.std():.3f} min={arr.min():.3f} max={arr.max():.3f} n={arr.size}")
    else:
        print(f"[OK] {ok}/{len(bases)} passed.")

# Check the most recent few:
check_files("train", recent_train, prefer="node")


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_6312\2499401837.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  g_src = torch.load(src, map_location="cpu")
C:\Users\MSHU

[WARN] shard_2c10.pt: no edge_index found on node (check your edge keys)
[WARN] shard_2c0e.pt: no edge_index found on node (check your edge keys)
[WARN] shard_2c05.pt: no edge_index found on node (check your edge keys)
[WARN] shard_2bb6.pt: no edge_index found on node (check your edge keys)
[WARN] shard_2ba8.pt: no edge_index found on node (check your edge keys)
[WARN] shard_2ba1.pt: no edge_index found on node (check your edge keys)
[WARN] shard_2b7e.pt: no edge_index found on node (check your edge keys)
[WARN] shard_2b77.pt: no edge_index found on node (check your edge keys)
[OK] 8/8 passed. Norms: mean=10.471 std=1.129 min=8.760 max=16.595 n=112717


Verify edges exits 

In [26]:
import torch
from pathlib import Path

p = next(Path("Dataset/train/hetero_ready_gcbert").glob("*.pt"))
g = torch.load(p, map_location="cpu")

print("node_types:", getattr(g, "node_types", ()))
print("edge_types:", getattr(g, "edge_types", ()))
for et in getattr(g, "edge_types", ()):
    ei = g[et].edge_index
    print(f"{et}: E={ei.size(1)}")


node_types: ['node']
edge_types: [('node', 'AST', 'node'), ('node', 'CFG', 'node'), ('node', 'DFG', 'node'), ('node', 'CALL', 'node'), ('node', 'ARG2PARAM', 'node'), ('node', 'RET2CALL', 'node'), ('node', 'RET2LHS', 'node'), ('node', 'AST_REV', 'node'), ('node', 'CFG_REV', 'node'), ('node', 'DFG_REV', 'node'), ('node', 'CALL_REV', 'node'), ('node', 'ARG2PARAM_REV', 'node'), ('node', 'RET2CALL_REV', 'node'), ('node', 'RET2LHS_REV', 'node')]
('node', 'AST', 'node'): E=2470
('node', 'CFG', 'node'): E=1457
('node', 'DFG', 'node'): E=697
('node', 'CALL', 'node'): E=608
('node', 'ARG2PARAM', 'node'): E=1097
('node', 'RET2CALL', 'node'): E=608
('node', 'RET2LHS', 'node'): E=96
('node', 'AST_REV', 'node'): E=2470
('node', 'CFG_REV', 'node'): E=1457
('node', 'DFG_REV', 'node'): E=697
('node', 'CALL_REV', 'node'): E=608
('node', 'ARG2PARAM_REV', 'node'): E=1097
('node', 'RET2CALL_REV', 'node'): E=608
('node', 'RET2LHS_REV', 'node'): E=96


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_6312\1908343408.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  g = torch.load(p, map_location="cpu")


In [80]:
import torch, json
from pathlib import Path

pt = next(Path("Dataset/train/hetero_ready_gcbert").glob("*.pt"))
g  = torch.load(pt, map_location="cpu")
nid = g["node"].nid[:10].tolist()
base = pt.stem
j   = json.load(open(f"Dataset/train/unified_aug/{base}.json","r",encoding="utf-8"))
json_ids = [n["_id"] for n in j["nodes"][:10]]
print("graph nid[:10]   =", nid)
print("json  _id[:10]   =", json_ids)

graph nid[:10]   = [21474836480, 21474836481, 21474836482, 21474836483, 21474836484, 21474836485, 21474836486, 21474836487, 21474836488, 21474836489]
json  _id[:10]   = [21474836480, 21474836481, 21474836482, 21474836483, 21474836484, 21474836485, 21474836486, 21474836487, 21474836488, 21474836489]


In [81]:
import json, torch
from pathlib import Path

SPLIT = "train"  # change to "valid"/"test" if needed
PT_DIR = Path(f"Dataset/{SPLIT}/hetero_ready_gcbert")
J_DIRS = [Path(f"Dataset/{SPLIT}/unified_aug"), Path(f"Dataset/{SPLIT}/unified")]

def _find_json(base):
    for jd in J_DIRS:
        for suf in (".json",".aug.json",".unified.json",".jsonl"):
            p = jd / f"{base}{suf}"
            if p.exists(): return p
        ms = list(jd.glob(f"{base}.*"))
        if ms: return ms[0]
    return None

def _coerce_int(x):
    try: return int(x)
    except Exception: return x

pts = sorted(PT_DIR.glob("*.pt"))
if not pts:
    print(f"No .pt files in {PT_DIR}")
else:
    total = ok = replaced = missing_json = 0
    examples = []
    for pt in pts:
        base = pt.stem
        jp = _find_json(base)
        if not jp:
            missing_json += 1
            continue

        g = torch.load(pt, map_location="cpu")
        st = g["node"]
        nid = st.nid.view(-1).tolist() if hasattr(st,"nid") else None

        try:
            raw = jp.read_text(encoding="utf-8")
            j = json.loads(raw)
        except Exception:
            print(f"[{base}] JSON parse error"); continue

        nodes = j.get("nodes", [])
        json_ids = [ _coerce_int(n.get("_id")) for n in nodes ]

        total += 1
        if nid is None:
            examples.append((base, "no_nid", None, json_ids[:5]))
            continue

        # element-wise check (same order) and set-coverage check
        same_len = len(nid)==len(json_ids)
        eq_order = same_len and nid == json_ids
        set_cov  = len(set(nid).intersection(set(json_ids))) / max(1, len(set(nid)))

        if eq_order:
            ok += 1
        else:
            # collect one mismatch example
            if len(examples) < 5:
                # show first 10 pairs to eyeball pattern
                pairs = list(zip(nid[:10], json_ids[:10]))
                examples.append((base, f"order_match={eq_order}, set_cov={set_cov:.3f}", pairs, None))

    print(f"[{SPLIT}] files: {len(pts)}  with_json: {total+missing_json}  json_missing: {missing_json}")
    print(f"Aligned (order match): {ok}/{total} ({(ok/max(1,total))*100:.1f}%)")
    print("Examples:")
    for ex in examples:
        base, note, pairs, jfirst = ex
        print(f" - {base}: {note}")
        if pairs: print("   nid vs json[:10] ->", pairs)
        if jfirst is not None: print("   json[:5] ->", jfirst)


[shard_336a] JSON parse error
[shard_cf65] JSON parse error
[train] files: 3438  with_json: 3436  json_missing: 0
Aligned (order match): 3436/3436 (100.0%)
Examples:


Fix the missing err in the trainin split

In [67]:
# fix_two_bad_train_gcbert.py
# Find the two failing TRAIN shards and regenerate ONLY those GC-BERT files.

import os, json, re, warnings
from pathlib import Path
from typing import Optional, List

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from transformers.utils import logging as hf_logging
from torch_geometric.data import HeteroData

# --- paths (TRAIN only) ---
IN_DIR   = Path("Dataset/train/hetero_ready")
JSON_DIRS= [Path("Dataset/train/unified_aug"), Path("Dataset/train/unified")]
OUT_DIR  = Path("Dataset/train/hetero_ready_gcbert")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- model config ---
MODEL_NAME = "microsoft/graphcodebert-base"
MAX_LEN    = 256
BATCH      = 32
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
EMB_DIM    = 768

# --- quiet logs ---
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`", category=FutureWarning)

# ---------------- helpers ----------------
def _as_hetero(x) -> HeteroData:
    if isinstance(x, HeteroData): return x
    raise TypeError(f"Expected HeteroData, got {type(x)}")

def _choose_node_store(g: HeteroData) -> str:
    nts = tuple(getattr(g, "node_types", ()) or [])
    for cand in ("node","ast","code","token","statement"):
        if cand in nts: return cand
    return nts[0] if nts else next(iter(g.keys()))

def _strip_comments_commas(txt: str) -> str:
    txt = re.sub(r'//.*?$', '', txt, flags=re.M)
    txt = re.sub(r'/\*.*?\*/', '', txt, flags=re.S)
    txt = txt.replace('\ufeff','').replace('\x00','')
    txt = re.sub(r',\s*(\}|\])', r'\1', txt)
    return txt

def _extract_nodes_array(txt: str) -> Optional[str]:
    m = re.search(r'"nodes"\s*:', txt)
    if not m: return None
    i = m.end()
    while i < len(txt) and txt[i] != '[': i += 1
    if i>=len(txt) or txt[i] != '[': return None
    depth=0; start=i
    for j,ch in enumerate(txt[i:], start=i):
        if ch=='[': depth+=1
        elif ch==']':
            depth-=1
            if depth==0: return txt[start:j+1]
    return None

def load_aug_json_robust(p: Path) -> Optional[dict]:
    if not p or not p.exists(): return None
    # direct
    try:
        return json.loads(p.read_text("utf-8"))
    except Exception:
        pass
    # stripped/repaired
    try:
        san = _strip_comments_commas(p.read_text("utf-8", errors="ignore"))
        return json.loads(san)
    except Exception:
        pass
    # nodes array only
    try:
        san = _strip_comments_commas(p.read_text("utf-8", errors="ignore"))
        arr = _extract_nodes_array(san)
        if arr:
            nds = json.loads(arr)
            if isinstance(nds, list):
                return {"nodes": nds}
    except Exception:
        pass
    # jsonl-ish fallback
    try:
        san = _strip_comments_commas(p.read_text("utf-8", errors="ignore"))
        nodes = []
        for line in san.splitlines():
            line=line.strip()
            if not line or line[0] not in "{[": continue
            try:
                o = json.loads(line)
                if isinstance(o, dict) and isinstance(o.get("nodes"), list): nodes += o["nodes"]
                elif isinstance(o, list): nodes += o
            except Exception:
                continue
        if nodes:
            return {"nodes": nodes}
    except Exception:
        pass
    return None

def find_json(base: str) -> Optional[Path]:
    for d in JSON_DIRS:
        for suf in (".json",".aug.json",".unified.json",".jsonl",".txt"):
            p = d / f"{base}{suf}"
            if p.exists(): return p
        ms = list(d.glob(f"{base}.*"))
        if ms: return ms[0]
    return None

def node_text_from_json(n: dict) -> str:
    for k in ("code", "name", "methodFullName", "signature", "text"):
        v = n.get(k)
        if isinstance(v, str) and v.strip(): return v.strip()
    return ""

@torch.no_grad()
def embed_gcbert(texts: List[str], tok, model) -> torch.Tensor:
    outs=[]; model.to(DEVICE).eval()
    total = (len(texts) + BATCH - 1)//BATCH
    for i in tqdm(range(0, len(texts), BATCH), total=total, desc="Embed", unit="batch", leave=False):
        chunk = texts[i:i+BATCH]
        enc = tok(chunk, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k,v in enc.items()}
        if DEVICE == "cuda":
            with torch.amp.autocast('cuda', dtype=torch.float16):
                hs = model(**enc).last_hidden_state
        else:
            hs = model(**enc).last_hidden_state
        outs.append(hs[:,0,:].contiguous().float().cpu())
    return torch.cat(outs, dim=0) if outs else torch.empty(0, EMB_DIM)

# ---------- detect the exact failing shards ----------
def detect_failed_train_shards() -> List[str]:
    bases_in  = {p.stem for p in IN_DIR.glob("*.pt")}
    bases_out = {p.stem for p in OUT_DIR.glob("*.pt")}
    # 1) missing in OUT_DIR
    missing = sorted(list(bases_in - bases_out))

    # 2) malformed outputs (bad or mismatched x_text) — recheck briefly
    malformed = []
    for p in OUT_DIR.glob("*.pt"):
        base = p.stem
        try:
            g = _as_hetero(torch.load(p, map_location="cpu"))
            st = g[_choose_node_store(g)]
            N  = st.num_nodes
            xt = getattr(st, "x_text", None)
            if xt is None or not torch.is_tensor(xt) or xt.size(0) != N:
                malformed.append(base)
        except Exception:
            malformed.append(base)

    # If you saw exactly 2 errs, this should give you those two basenames.
    failed = sorted(list(set(missing + malformed)))
    # Keep it tiny: if more than 2 surfaces for any reason, still fix all failed found.
    return failed

# ---------- regenerate only failed ----------
def regenerate_failed_only():
    failed = detect_failed_train_shards()
    if not failed:
        print("No failing TRAIN shards detected. Nothing to fix.")
        return

    print("Will fix these TRAIN shard(s):", failed)

    tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    mdl = AutoModel.from_pretrained(MODEL_NAME, use_safetensors=True)

    fixed = 0
    for base in tqdm(failed, desc="Fix+Save", unit="file"):
        in_pt = IN_DIR / f"{base}.pt"
        jp    = find_json(base)

        # Load input graph
        try:
            g = _as_hetero(torch.load(in_pt, map_location="cpu"))
        except Exception as e:
            print(f"[ERR] {base}: cannot load input ({e})")
            continue

        st   = g[_choose_node_store(g)]
        N    = st.num_nodes

        # Load JSON to get texts aligned to node IDs (nid if present; else by index)
        j = load_aug_json_robust(jp) if jp else None
        nodes = j.get("nodes", []) if isinstance(j, dict) else (j if isinstance(j, list) else [])

        if hasattr(st, "nid"):
            # map nid -> json node (by _id), fallback to empty text
            id2node = {}
            for n in nodes:
                if isinstance(n, dict) and "_id" in n:
                    id2node[int(n["_id"])] = n
            texts = []
            for nid in st.nid.view(-1).tolist():
                nd = id2node.get(int(nid))
                texts.append(node_text_from_json(nd) if isinstance(nd, dict) else "<node>")
        else:
            # fallback: assume same order as JSON nodes
            if isinstance(nodes, list) and len(nodes) == N:
                texts = [node_text_from_json(n) if isinstance(n, dict) else "<node>" for n in nodes]
                st.nid = torch.arange(N, dtype=torch.long)  # create a simple nid to stabilize
            else:
                # ultimate fallback: no JSON or length mismatch
                texts = ["<node>"] * N
                if not hasattr(st, "nid"):
                    st.nid = torch.arange(N, dtype=torch.long)

        # Embed & save
        x_text = embed_gcbert(texts, tok, mdl)
        if x_text.size(0) != N:
            print(f"[ERR] {base}: embed size mismatch ({x_text.size(0)} vs {N}); skipping.")
            continue

        st.x_text = x_text
        try:
            torch.save(g, OUT_DIR / f"{base}.pt")
            fixed += 1
        except Exception as e:
            print(f"[ERR] {base}: save failed ({e})")
            continue

    print(f"Done. Fixed & regenerated: {fixed} shard(s) → {OUT_DIR}")

# ---- run just this once ----
if __name__ == "__main__":
    regenerate_failed_only()


Will fix these TRAIN shard(s): ['shard_336a', 'shard_cf65']


Fix+Save: 100%|██████████| 2/2 [00:42<00:00, 21.48s/file]

Done. Fixed & regenerated: 2 shard(s) → Dataset\train\hetero_ready_gcbert


In [82]:
# %% [markdown]
# GC-BERT Sanity Checker (run this cell)
# - Scans Dataset/<split>/hetero_ready_gcbert
# - Verifies x_text shape, nid presence, JSON alignment, inter-procedural edges
# - Writes per-file JSONL and an aggregate summary with problem shard lists

# %%
import os, re, json, warnings
from pathlib import Path
from collections import Counter
from typing import Optional

import torch
from tqdm.auto import tqdm
from torch_geometric.data import HeteroData

# ---------------- Config ----------------
SPLIT = "train"  # change to "valid"/"test" if needed

IN_DIR      = Path(f"Dataset/{SPLIT}/hetero_ready_gcbert")
RAW_DIR     = Path(f"Dataset/{SPLIT}/hetero_ready")  # unused but handy for manual checks
JSON_DIRS   = [Path(f"Dataset/{SPLIT}/unified_aug"), Path(f"Dataset/{SPLIT}/unified")]
OUT_DIR     = Path("out/gcbert_sanity"); OUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_JSONL = OUT_DIR / f"{SPLIT}_report.jsonl"
SUMMARY_JSON = OUT_DIR / f"{SPLIT}_summary.json"

# ---------------- Quiet warnings ----------------
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`", category=FutureWarning)

# ---------------- Helpers ----------------
def _as_hetero(x) -> HeteroData:
    if isinstance(x, HeteroData): return x
    raise TypeError(f"Expected HeteroData, got {type(x)}")

def _choose_node_store(g: HeteroData) -> str:
    nts = tuple(getattr(g, "node_types", ()) or [])
    for cand in ("node","ast","code","token","statement"):
        if cand in nts: return cand
    return nts[0] if nts else next(iter(g.keys()))

def _strip_comments_commas(txt: str) -> str:
    txt = re.sub(r'//.*?$', '', txt, flags=re.M)
    txt = re.sub(r'/\*.*?\*/', '', txt, flags=re.S)
    txt = txt.replace('\ufeff','').replace('\x00','')
    txt = re.sub(r',\s*(\}|\])', r'\1', txt)
    return txt

def _extract_nodes_array(txt: str) -> Optional[str]:
    m = re.search(r'"nodes"\s*:', txt)
    if not m: return None
    i = m.end()
    while i < len(txt) and txt[i] != '[': i += 1
    if i>=len(txt) or txt[i] != '[': return None
    depth=0; start=i
    for j,ch in enumerate(txt[i:], start=i):
        if ch=='[': depth+=1
        elif ch==']':
            depth-=1
            if depth==0: return txt[start:j+1]
    return None

def load_aug_json_robust(p: Optional[Path]) -> Optional[dict | list]:
    if p is None or not p.exists(): return None
    # direct
    try:
        return json.loads(p.read_text("utf-8"))
    except Exception:
        pass
    # stripped/repaired
    try:
        san = _strip_comments_commas(p.read_text("utf-8", errors="ignore"))
        return json.loads(san)
    except Exception:
        pass
    # nodes array only
    try:
        san = _strip_comments_commas(p.read_text("utf-8", errors="ignore"))
        arr = _extract_nodes_array(san)
        if arr:
            nds = json.loads(arr)
            if isinstance(nds, list):
                return {"nodes": nds}
    except Exception:
        pass
    # jsonl-ish fallback
    try:
        san = _strip_comments_commas(p.read_text("utf-8", errors="ignore"))
        nodes = []
        for line in san.splitlines():
            line=line.strip()
            if not line or line[0] not in "{[": continue
            try:
                o = json.loads(line)
                if isinstance(o, dict) and isinstance(o.get("nodes"), list): nodes += o["nodes"]
                elif isinstance(o, list): nodes += o
            except Exception:
                continue
        if nodes:
            return {"nodes": nodes}
    except Exception:
        pass
    return None

def find_json(base: str) -> Optional[Path]:
    for d in JSON_DIRS:
        for suf in (".json",".aug.json",".unified.json",".jsonl",".txt"):
            p = d / f"{base}{suf}"
            if p.exists(): return p
        ms = list(d.glob(f"{base}.*"))
        if ms: return ms[0]
    return None

def _coerce_int(x):
    if isinstance(x, int): return x
    if isinstance(x, str) and x.strip().lstrip("-").isdigit():
        try: return int(x)
        except ValueError: return x
    return x

def harvest_positive_ids(j) -> set[int]:
    """Collect positive node ids from any JSON shape (explicit labels, alt keys, path endpoints)."""
    pos = set()
    if j is None: return pos
    # nodes with labels
    nodes = []
    if isinstance(j, dict) and isinstance(j.get("nodes"), list):
        nodes = j["nodes"]
    elif isinstance(j, list):
        nodes = j
    for n in nodes:
        if not isinstance(n, dict): continue
        nid = _coerce_int(n.get("_id"))
        lab = (n.get("label") or n.get("_label") or n.get("class") or n.get("tag") or "")
        lab_u = str(lab).upper()
        is_sink = bool(n.get("is_sink")) or ("SINK" in lab_u) or ("VULN" in lab_u) or ("VULNERABLE" in lab_u)
        if isinstance(nid, int) and is_sink:
            pos.add(nid)
    # alt keys
    if isinstance(j, dict):
        for k in ("sinks","vulnerabilities","vul_nodes","labels","positives","positive_nodes"):
            arr = j.get(k)
            if isinstance(arr, list):
                for x in arr:
                    xi = _coerce_int(x)
                    if isinstance(xi, int): pos.add(xi)
        # endpoints of vulnerable_paths
        vps = j.get("vulnerable_paths", [])
        if isinstance(vps, list):
            for p in vps:
                if isinstance(p, (list, tuple)) and len(p)>0:
                    a = _coerce_int(p[0]); b = _coerce_int(p[-1])
                    if isinstance(a, int): pos.add(a)
                    if isinstance(b, int): pos.add(b)
    return pos

def inter_edges_stats(g: HeteroData) -> dict:
    from collections import Counter
    stats = Counter()
    for (s,r,t) in g.edge_types:
        if r in ("CALL","ARG2PARAM","RET2CALL","RET2LHS","CALL_REV","ARG2PARAM_REV","RET2CALL_REV","RET2LHS_REV"):
            ei = getattr(g[(s,r,t)], "edge_index", None)
            stats[r] += 0 if ei is None else int(ei.size(1))
    return dict(stats)

# ---------------- Main pass ----------------
# clean previous report
if REPORT_JSONL.exists():
    REPORT_JSONL.unlink()

pts = sorted(IN_DIR.glob("*.pt"))
if not pts:
    raise FileNotFoundError(f"No .pt files in {IN_DIR}")

totals = Counter()
bad = {
    "missing_json": [],
    "bad_json": [],
    "no_nid": [],
    "x_text_missing": [],
    "x_text_size_mismatch": [],
    "no_inter_edges": [],
    "no_positive_ids_in_json": [],
    "json_ids_not_in_nid": [],
    "zero_positives_after_mapping": [],
    "load_failed": [],
}

with open(REPORT_JSONL, "a", encoding="utf-8") as out_f:
    for pt in tqdm(pts, desc=f"{SPLIT.upper()} sanity", unit="file"):
        base = pt.stem

        # Load graph
        try:
            g = _as_hetero(torch.load(pt, map_location="cpu"))
            st_name = _choose_node_store(g)
            st = g[st_name]
            N = st.num_nodes
        except Exception as e:
            out_f.write(json.dumps({"file": base, "error": f"load_failed: {e}"}) + "\n")
            bad["load_failed"].append(base)
            continue

        # Check x_text
        xt = getattr(st, "x_text", None)
        if xt is None:
            bad["x_text_missing"].append(base)
        elif not torch.is_tensor(xt) or xt.ndim != 2 or xt.size(0) != N:
            bad["x_text_size_mismatch"].append(base)

        # JSON
        jp = find_json(base)
        if jp is None:
            bad["missing_json"].append(base)
            j = None
        else:
            j = load_aug_json_robust(jp)
            if j is None:
                bad["bad_json"].append(base)

        # nid
        has_nid = hasattr(st, "nid")
        if not has_nid:
            bad["no_nid"].append(base)
            nid_list = list(range(N))
        else:
            nid_list = st.nid.view(-1).tolist()

        # JSON ids & mapping
        pos_ids_json = harvest_positive_ids(j) if j is not None else set()
        totals["files"] += 1
        totals["nodes"] += N
        totals["with_json"] += int(j is not None)
        totals["with_nid"] += int(has_nid)
        totals["xt_ok"] += int(xt is not None and torch.is_tensor(xt) and xt.size(0) == N)

        inter_stats = inter_edges_stats(g)
        if sum(inter_stats.values()) == 0:
            bad["no_inter_edges"].append(base)

        nid_set = set(int(x) for x in nid_list)
        found_ids = [i for i in pos_ids_json if i in nid_set]
        not_found = [i for i in pos_ids_json if i not in nid_set]

        if j is not None and len(pos_ids_json) == 0:
            bad["no_positive_ids_in_json"].append(base)
        if j is not None and len(pos_ids_json) > 0 and len(found_ids) == 0:
            bad["json_ids_not_in_nid"].append(base)
        if len(found_ids) == 0:
            bad["zero_positives_after_mapping"].append(base)

        line = {
            "file": base,
            "num_nodes": N,
            "x_text_ok": bool(xt is not None and torch.is_tensor(xt) and xt.size(0)==N),
            "has_nid": has_nid,
            "json_present": j is not None,
            "json_pos_ids": len(pos_ids_json),
            "json_pos_ids_in_nid": len(found_ids),
            "json_pos_ids_not_in_nid": len(not_found),
            "any_inter_edges": sum(inter_stats.values()) > 0,
            "inter_stats": inter_stats,
        }
        out_f.write(json.dumps(line) + "\n")

summary = {
    "split": SPLIT,
    "counts": dict(totals),
    "problems": {k: sorted(set(v)) for k,v in bad.items() if v},
    "suggestions": [
        "For shards in 'json_ids_not_in_nid': JSON _id space ≠ graph.nid. Regenerate or remap nid to Joern ids.",
        "For 'x_text_missing'/'x_text_size_mismatch': re-embed those shards only.",
        "For 'no_inter_edges': CALL/ARG2PARAM/RET edges missing → no inter-procedural learning.",
        "If many 'no_positive_ids_in_json': confirm your JSON labels or vulnerable_paths.",
    ],
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("\n=== GC-BERT Sanity Summary ===")
print(f"Files scanned     : {totals['files']}")
print(f"x_text OK         : {totals['xt_ok']} / {totals['files']}")
print(f"have nid          : {totals['with_nid']} / {totals['files']}")
print(f"have JSON         : {totals['with_json']} / {totals['files']}")
print("Problem categories with counts (see summary JSON for filenames):")
for k, arr in summary["problems"].items():
    print(f"  - {k:28s}: {len(arr)}")

print(f"\nPer-file report : {REPORT_JSONL}")
print(f"Aggregate summary: {SUMMARY_JSON}")


TRAIN sanity: 100%|██████████| 3438/3438 [03:31<00:00, 16.25file/s]


=== GC-BERT Sanity Summary ===
Files scanned     : 3438
x_text OK         : 3438 / 3438
have nid          : 3438 / 3438
have JSON         : 3436 / 3438
Problem categories with counts (see summary JSON for filenames):
  - bad_json                    : 2
  - no_inter_edges              : 26
  - no_positive_ids_in_json     : 3436
  - zero_positives_after_mapping: 3438

Per-file report : out\gcbert_sanity\train_report.jsonl
Aggregate summary: out\gcbert_sanity\train_summary.json


In [83]:
# %% [markdown]
# 1) JSON Schema Auditor — run this first
# Finds how positives are represented:
# - looks for keys like is_sink / isVulnerable / vulnerable / label/class/tag
# - inspects vulnerable_paths element types (int/str/dict)
# Writes a compact report with examples so we can wire the labeler correctly.

# %%
import json, re
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm

SPLIT = "train"
JSON_DIRS = [Path(f"Dataset/{SPLIT}/unified_aug"), Path(f"Dataset/{SPLIT}/unified")]
OUT_DIR = Path("out/gcbert_sanity"); OUT_DIR.mkdir(parents=True, exist_ok=True)
SCHEMA_JSON = OUT_DIR / f"{SPLIT}_json_schema_audit.json"

def _load_robust(p: Path):
    try:
        return json.loads(p.read_text("utf-8"))
    except Exception:
        pass
    # strip comments/commas
    try:
        txt = p.read_text("utf-8", errors="ignore")
        txt = re.sub(r'//.*?$', '', txt, flags=re.M)
        txt = re.sub(r'/\*.*?\*/', '', txt, flags=re.S)
        txt = re.sub(r',\s*(\}|\])', r'\1', txt)
        return json.loads(txt)
    except Exception:
        return None

def _find(base: str):
    for d in JSON_DIRS:
        for suf in (".json",".aug.json",".unified.json",".jsonl",".txt"):
            p = d / f"{base}{suf}"
            if p.exists(): return p
        ms = list(d.glob(f"{base}.*"))
        if ms: return ms[0]
    return None

# scan
stats = Counter()
key_hits = Counter()
path_elem_types = Counter()
examples = defaultdict(list)

pt_dir = Path(f"Dataset/{SPLIT}/hetero_ready_gcbert")
for pt in tqdm(sorted(pt_dir.glob("*.pt")), desc="Audit JSON schema", unit="file"):
    base = pt.stem
    jp = _find(base)
    if not jp:
        stats["missing_json"] += 1
        continue
    j = _load_robust(jp)
    if j is None:
        stats["bad_json"] += 1
        continue

    stats["json_ok"] += 1

    # common positive patterns
    nodes = j.get("nodes") if isinstance(j, dict) else (j if isinstance(j, list) else None)
    if isinstance(nodes, list):
        stats["has_nodes"] += 1
        for n in nodes[:50]:  # sample
            if isinstance(n, dict):
                for k in ("is_sink","isVulnerable","vulnerable","label","_label","class","tag"):
                    if k in n: key_hits[f"nodes.{k}"] += 1
                    if len(examples[f"nodes.{k}"]) < 3 and k in n:
                        examples[f"nodes.{k}"].append({k: n[k]})
    else:
        stats["no_nodes_array"] += 1

    # alt positive keys
    if isinstance(j, dict):
        for k in ("sinks","vulnerabilities","vul_nodes","labels","positives","positive_nodes"):
            if k in j:
                key_hits[k] += 1
                if len(examples[k]) < 3:
                    examples[k].append({"sample": j[k][:5] if isinstance(j[k], list) else j[k]})

        # vulnerable_paths shapes
        vps = j.get("vulnerable_paths")
        if isinstance(vps, list):
            stats["has_vulnerable_paths"] += 1
            for p in vps[:20]:
                if isinstance(p, (list, tuple)) and p:
                    for elem in p[:5]:
                        t = type(elem).__name__
                        if isinstance(elem, dict):
                            # capture common fields
                            t += ":" + ",".join(sorted(set(elem.keys()) & {"_id","id","joernId","nodeId","name","kind"}))
                            if len(examples["vulnerable_paths_elem_dict"]) < 5:
                                examples["vulnerable_paths_elem_dict"].append(elem)
                        path_elem_types[t] += 1
        else:
            stats["no_vulnerable_paths"] += 1

audit = {
    "split": SPLIT,
    "stats": dict(stats),
    "key_hits": dict(key_hits.most_common()),
    "vulnerable_paths_elem_types": dict(path_elem_types.most_common()),
    "examples": {k:v for k,v in examples.items()},
}
SCHEMA_JSON.write_text(json.dumps(audit, indent=2), encoding="utf-8")
print(f"Wrote schema audit → {SCHEMA_JSON}")
print("Open it and look for where positives live (keys used, path element types).")


Audit JSON schema: 100%|██████████| 3438/3438 [02:32<00:00, 22.57file/s]

Wrote schema audit → out\gcbert_sanity\train_json_schema_audit.json
Open it and look for where positives live (keys used, path element types).


In [ ]:
# %% [markdown]
# 2) Labels Cache Builder — run after reviewing the schema audit.
# It extracts positive IDs per shard using robust rules and writes:
#   out/labels_cache/train_labels.jsonl  (one line per file: {"file": <base>, "pos": [ids...]})
# It also optionally expands positives with inter-procedural K-hop neighborhood.

# %%
import json, re
from pathlib import Path
from collections import deque
from tqdm.auto import tqdm
import torch
from torch_geometric.data import HeteroData

SPLIT = "train"
JSON_DIRS = [Path(f"Dataset/{SPLIT}/unified_aug"), Path(f"Dataset/{SPLIT}/unified")]
PT_DIR   = Path(f"Dataset/{SPLIT}/hetero_ready_gcbert")
OUT_DIR  = Path("out/labels_cache"); OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSONL = OUT_DIR / f"{SPLIT}_labels.jsonl"

# set how far to expand along inter-procedural edges (CALL/ARG2PARAM/RET*)
K_HOP_INTER = 2  # set 0 to disable expansion

def _load_robust(p: Path):
    try:
        return json.loads(p.read_text("utf-8"))
    except Exception:
        pass
    try:
        txt = p.read_text("utf-8", errors="ignore")
        txt = re.sub(r'//.*?$', '', txt, flags=re.M)
        txt = re.sub(r'/\*.*?\*/', '', txt, flags=re.S)
        txt = re.sub(r',\s*(\}|\])', r'\1', txt)
        return json.loads(txt)
    except Exception:
        return None

def _find(base: str):
    for d in JSON_DIRS:
        for suf in (".json",".aug.json",".unified.json",".jsonl",".txt"):
            p = d / f"{base}{suf}"
            if p.exists(): return p
        ms = list(d.glob(f"{base}.*"))
        if ms: return ms[0]
    return None

def _coerce_int(x):
    if isinstance(x, int): return x
    if isinstance(x, str) and x.strip().lstrip("-").isdigit():
        try: return int(x)
        except ValueError: return x
    return x

def harvest_pos_ids(j) -> set[int]:
    pos = set()
    if j is None: return pos

    # explicit node-level flags
    nodes = j.get("nodes") if isinstance(j, dict) else (j if isinstance(j, list) else None)
    if isinstance(nodes, list):
        for n in nodes:
            if not isinstance(n, dict): continue
            nid = _coerce_int(n.get("_id") or n.get("id") or n.get("joernId") or n.get("nodeId"))
            # expand the set of flags based on the schema audit if needed
            lab = (n.get("label") or n.get("_label") or n.get("class") or n.get("tag") or "")
            lab_u = str(lab).upper()
            is_pos = (
                bool(n.get("is_sink")) or bool(n.get("isVulnerable")) or bool(n.get("vulnerable")) or
                ("SINK" in lab_u) or ("VULN" in lab_u) or ("VULNERABLE" in lab_u)
            )
            if isinstance(nid, int) and is_pos:
                pos.add(nid)

    # alt top-level keys
    if isinstance(j, dict):
        for k in ("sinks","vulnerabilities","vul_nodes","labels","positives","positive_nodes"):
            arr = j.get(k)
            if isinstance(arr, list):
                for x in arr:
                    xi = _coerce_int(x)
                    if isinstance(xi, int):
                        pos.add(xi)

        # vulnerable_paths endpoints & elements
        vps = j.get("vulnerable_paths")
        if isinstance(vps, list):
            for p in vps:
                if not isinstance(p, (list, tuple)): continue
                for elem in p:
                    # handle int / str / dict cases
                    if isinstance(elem, int) or (isinstance(elem, str) and elem.strip().lstrip("-").isdigit()):
                        xi = _coerce_int(elem)
                        if isinstance(xi, int): pos.add(xi)
                    elif isinstance(elem, dict):
                        for kk in ("_id","id","joernId","nodeId"):
                            if kk in elem:
                                xi = _coerce_int(elem[kk])
                                if isinstance(xi, int): pos.add(xi)
                                break
                    # nested list? flatten a bit
                    elif isinstance(elem, (list, tuple)):
                        for sub in elem:
                            xi = _coerce_int(sub)
                            if isinstance(xi, int): pos.add(xi)
    return pos

def get_edge_index(g: HeteroData, et):
    if et not in g.edge_types: return None
    ei = getattr(g[et], "edge_index", None)
    return ei

def expand_interprocedural(g: HeteroData, base_pos: set[int], k: int) -> set[int]:
    if k <= 0 or not base_pos: return set(base_pos)
    # build undirected adjacency over inter-proc relations
    inter_rel = {
        'CALL','ARG2PARAM','RET2CALL','RET2LHS',
        'CALL_REV','ARG2PARAM_REV','RET2CALL_REV','RET2LHS_REV'
    }
    N = g['node'].num_nodes
    adj = [[] for _ in range(N)]
    for (s,r,t) in g.edge_types:
        if r in inter_rel:
            ei = get_edge_index(g, (s,r,t))
            if ei is None: continue
            u, v = ei
            for a,b in zip(u.tolist(), v.tolist()):
                adj[a].append(b); adj[b].append(a)

    visited = set(base_pos)
    q = deque([(u,0) for u in base_pos])
    while q:
        u, d = q.popleft()
        if d >= k: continue
        for v in adj[u]:
            if v not in visited:
                visited.add(v)
                q.append((v, d+1))
    return visited

# build cache
if OUT_JSONL.exists():
    OUT_JSONL.unlink()

written = skipped = 0
for pt in tqdm(sorted(PT_DIR.glob("*.pt")), desc="Build labels cache", unit="file"):
    base = pt.stem
    jp = _find(base)
    j  = _load_robust(jp) if jp else None

    # load graph for nid & expansion
    g = torch.load(pt, map_location="cpu")
    st = g["node"]
    nid = st.nid.view(-1).tolist()

    pos_ids = harvest_pos_ids(j)
    # map json ids -> graph positions via nid
    nid2pos = {int(n): i for i, n in enumerate(nid)}
    pos_idx = [nid2pos[i] for i in pos_ids if int(i) in nid2pos]

    # optional inter-procedural expansion (k hops)
    if pos_idx and K_HOP_INTER > 0:
        pos_idx = list(expand_interprocedural(g, set(pos_idx), K_HOP_INTER))

    record = {"file": base, "pos": sorted(set(int(i) for i in pos_idx))}
    if record["pos"]:
        with open(OUT_JSONL, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
        written += 1
    else:
        skipped += 1

print(f"labels cache → {OUT_JSONL}  (graphs with labels: {written}, skipped: {skipped})")
print("Next: update your training loader to read this cache and build y tensors.")


Build labels cache:  36%|███▋      | 1247/3438 [01:11<01:44, 21.00file/s]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x0000020651213BD0>>
Traceback (most recent call last):
  File "c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\.venv\Lib\site-packages\ipykernel\ipkernel.py", line 796, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
                                                 ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\MSHUVO23\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1501, in enumerate
    def enumerate():
    
KeyboardInterrupt: 
Build labels cache:  42%|████▏     | 1443/3438 [01:23<01:51, 17.87file/s]

In [7]:
# =======================================
# CA-GAT + ACC (robust static seeds + inter-aware weak labels) — Training Cell
# =======================================
import os, re, json, math, atexit, traceback, warnings, random, time
from pathlib import Path
from collections import defaultdict, deque

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from packaging.version import parse as V
from tqdm.auto import tqdm

# PYG
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv

# ------------------------------ top-level config ------------------------------
SPLIT                 = "train"  # "train" | "valid" | "test"
GC_DIR                = f"Dataset/{SPLIT}/hetero_ready_gcbert"
UNIFIED_JSON_DIRS     = [f"Dataset/{SPLIT}/unified_json", f"Dataset/{SPLIT}/unified"]
AUG_JSON_DIRS         = [f"Dataset/{SPLIT}/unified_aug", f"Dataset/{SPLIT}/unified"]
LOGDIR                = f"out/cagat_acc_interproc_{SPLIT}"

NODE_TYPE             = "node"
DEVICE                = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CUDA & memory knobs
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

USE_AMP            = True   # mixed precision
USE_CHECKPOINT     = True   # gradient checkpointing in GAT blocks
CLEAR_CACHE_EVERY  = 25
SKIP_ON_OOM        = True

# Train hyperparams
EPOCHS                = 3
BATCH_SIZE            = 1
HIDDEN                = 128
HEADS                 = 4
LAYERS                = 4
DROPOUT               = 0.10
LR                    = 2e-3
WEIGHT_DECAY          = 1e-4
SAVE_EVERY_STEPS      = 200
LOG_JSONL_EVERY       = 10

# Call-chain contextualization
CALL_CHAIN_ATTN       = True
CALL_CHAIN_MAX_HOPS   = 3
CALL_CHAIN_SPARSE_NTHRESH = 5000

# Loss weights (scheduled)
LAMBDA_NODE_BCE_BASE       = 1.0
LAMBDA_EDGE_SMOOTH_BASE    = 0.05
LAMBDA_INTER_CONTRAST_BASE = 0.05
LAMBDA_PATH_SUPER_BASE     = 0.10

# Edge weights for smoothness
EDGE_WEIGHTS = {
    'CFG': 0.5, 'DFG': 0.8, 'CFG_REV': 0.3, 'DFG_REV': 0.5,
    'CALL': 1.5, 'CALL_REV': 1.0,
    'ARG2PARAM': 2.0, 'ARG2PARAM_REV': 1.5,
    'RET2CALL': 2.0, 'RET2CALL_REV': 1.5,
    'RET2LHS': 1.8, 'RET2LHS_REV': 1.2,
}

warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`", category=FutureWarning)

# ------------------------------ AMP handling (Torch 2.4.x clean) ------------------------------
_TVER = V(torch.__version__)
USE_NEW_AMP_API = _TVER >= V("2.5.0")
if USE_NEW_AMP_API:
    from torch.amp import autocast as _autocast_new, GradScaler as _GradScalerNew
    def autocast_ctx(enabled=True): 
        return _autocast_new(device_type=("cuda" if DEVICE.type=="cuda" else "cpu"),
                             dtype=torch.float16, enabled=enabled)
    SCALER = _GradScalerNew("cuda" if DEVICE.type=="cuda" else "cpu", enabled=(USE_AMP and DEVICE.type=="cuda"))
else:
    from torch.cuda.amp import autocast as _autocast_old, GradScaler as _GradScalerOld
    warnings.filterwarnings("ignore", category=FutureWarning, message="`torch.cuda.amp.autocast")
    warnings.filterwarnings("ignore", category=FutureWarning, message="`torch.cuda.amp.GradScaler")
    def autocast_ctx(enabled=True): 
        return _autocast_old(dtype=torch.float16, enabled=(enabled and DEVICE.type=="cuda"))
    SCALER = _GradScalerOld(enabled=(USE_AMP and DEVICE.type=="cuda"))
print(f"[AMP] torch={torch.__version__} device={DEVICE.type}")

# ------------------------------ IO helpers ------------------------------
def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def find_json(basename: str) -> str | None:
    for d in AUG_JSON_DIRS:
        jd = Path(d)
        for suf in (".json",".aug.json",".unified.json",".jsonl",".txt"):
            p = jd / f"{basename}{suf}"
            if p.exists(): return str(p)
        ms = sorted(jd.glob(f"{basename}.*"))
        if ms: return str(ms[0])
    return None

def _strip_comments_commas(txt: str) -> str:
    txt = re.sub(r'//.*?$', '', txt, flags=re.M)
    txt = re.sub(r'/\*.*?\*/', '', txt, flags=re.S)
    txt = txt.replace('\ufeff','').replace('\x00','')
    txt = re.sub(r',\s*(\}|\])', r'\1', txt)
    return txt

def _extract_nodes_array(txt: str):
    m = re.search(r'"nodes"\s*:', txt)
    if not m: return None
    i = m.end()
    while i < len(txt) and txt[i] != '[': i += 1
    if i>=len(txt) or txt[i] != '[': return None
    depth=0; start=i
    for j,ch in enumerate(txt[i:], start=i):
        if ch=='[': depth+=1
        elif ch==']':
            depth-=1
            if depth==0: return txt[start:j+1]
    return None

def load_aug_json(path: str) -> dict | list | None:
    if not path or not os.path.exists(path): return None
    p = Path(path)
    for enc in ("utf-8","utf-8-sig","latin-1"):
        try:
            raw = p.read_text(enc)
            break
        except Exception:
            raw = None
    if raw is None: return None
    try:
        j = json.loads(raw); 
        if isinstance(j,(dict,list)): return j
    except json.JSONDecodeError: pass
    san = _strip_comments_commas(raw)
    try:
        j = json.loads(san); 
        if isinstance(j,(dict,list)): return j
    except json.JSONDecodeError: pass
    arr = _extract_nodes_array(san)
    if arr:
        try:
            nodes = json.loads(arr)
            if isinstance(nodes, list): return {"nodes": nodes}
        except json.JSONDecodeError: pass
    nodes=[]; paths=[]
    for line in san.splitlines():
        line=line.strip()
        if not line or line[0] not in "{[": continue
        try:
            o = json.loads(line)
            if isinstance(o, dict):
                if isinstance(o.get("nodes"), list): nodes += o["nodes"]
                if isinstance(o.get("vulnerable_paths"), list): paths += o["vulnerable_paths"]
                for k in ("sinks","vulnerabilities","vul_nodes","labels","positives"):
                    if isinstance(o.get(k), list):
                        nodes += [{'_id': x, 'is_sink': True} for x in o[k]]
            elif isinstance(o, list):
                nodes += o
        except json.JSONDecodeError:
            continue
    if nodes or paths:
        d={"nodes": nodes} if nodes else {}
        if paths: d["vulnerable_paths"]=paths
        return d if d else None
    return None

def _coerce_int(x):
    if isinstance(x, int): return x
    if isinstance(x, str) and x.strip().lstrip("-").isdigit():
        try: return int(x)
        except ValueError: return x
    return x

def _harvest_ids_from_any_json(j):
    pos = set()
    if j is None: return pos
    nodes = j["nodes"] if isinstance(j, dict) and isinstance(j.get("nodes"), list) else (j if isinstance(j, list) else [])
    for n in nodes:
        if not isinstance(n, dict): continue
        nid = _coerce_int(n.get("_id"))
        lab = (n.get("label") or n.get("_label") or n.get("class") or n.get("tag") or "")
        lab_u = str(lab).upper()
        is_sink = bool(n.get("is_sink")) or ("SINK" in lab_u) or ("VULN" in lab_u) or ("VULNERABLE" in lab_u)
        if isinstance(nid, int) and is_sink:
            pos.add(nid)
    if isinstance(j, dict):
        for k in ("sinks","vulnerabilities","vul_nodes","labels","positives","positive_nodes"):
            if isinstance(j.get(k), list):
                for x in j[k]:
                    xi = _coerce_int(x)
                    if isinstance(xi, int): pos.add(xi)
        vps = j.get("vulnerable_paths", [])
        if isinstance(vps, list):
            for p in vps:
                if isinstance(p, (list, tuple)) and len(p)>0:
                    a = _coerce_int(p[0]); b = _coerce_int(p[-1])
                    if isinstance(a, int): pos.add(a)
                    if isinstance(b, int): pos.add(b)
    return pos

def node_sink_labels_from_json(json_path: str, g: HeteroData, use_paths_as_pos=True) -> torch.Tensor | None:
    if not json_path or not os.path.exists(json_path): return None
    j = load_aug_json(json_path)
    if j is None: return None
    pos_ids = _harvest_ids_from_any_json(j)
    st = g[NODE_TYPE]
    device = (st.x_text.device if hasattr(st,"x_text") else (st.x.device if hasattr(st,"x") else "cpu"))
    if hasattr(st, "nid"):
        ids = [ _coerce_int(i) for i in st.nid.view(-1).tolist() ]
        return torch.tensor([ 1.0 if (isinstance(i,int) and i in pos_ids) else 0.0 for i in ids ],
                            dtype=torch.float32, device=device)
    if isinstance(j, dict) and isinstance(j.get("nodes"), list) and len(j["nodes"]) == st.num_nodes:
        ordered_ids = [ _coerce_int(n.get("_id")) if isinstance(n, dict) else None for n in j["nodes"] ]
        return torch.tensor([ 1.0 if (isinstance(i,int) and i in pos_ids) else 0.0 for i in ordered_ids ],
                            dtype=torch.float32, device=device)
    return None

def read_paths_from_json(json_path: str) -> list:
    if not json_path or not os.path.exists(json_path): return []
    j = load_aug_json(json_path)
    if not isinstance(j, dict): return []
    paths = j.get("vulnerable_paths", [])
    return paths if isinstance(paths, list) else []

# ------------------------------ edges / dataset ------------------------------
def get_edge_index(g: HeteroData, et: tuple[str,str,str]) -> torch.Tensor:
    if et not in g.edge_types:
        dev = (g[NODE_TYPE].x_text.device if hasattr(g[NODE_TYPE],"x_text")
               else (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else 'cpu'))
        return torch.zeros((2,0), dtype=torch.long, device=dev)
    store = g[et]
    ei = getattr(store, "edge_index", None)
    if ei is not None:
        return ei
    adj_t = getattr(store, "adj_t", None)
    if adj_t is not None:
        row, col, _ = adj_t.coo()     # adj_t is transposed
        return torch.stack([col, row], dim=0)
    dev = (g[NODE_TYPE].x_text.device if hasattr(g[NODE_TYPE],"x_text")
           else (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else 'cpu'))
    return torch.zeros((2,0), dtype=torch.long, device=dev)

class GraphDir(Dataset):
    def __init__(self, gc_dir: str):
        self.paths = sorted(Path(gc_dir).glob("*.pt"))
        if not self.paths:
            raise FileNotFoundError(f"No .pt in {gc_dir}")
        print(f"[DATA] {len(self.paths)} graphs in {gc_dir}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        pt = self.paths[i]
        g  = safe_load(pt, map_location="cpu")
        g.__dict__["_aug_json_path"] = find_json(pt.stem)
        return g

def validate_interprocedural_coverage(g: HeteroData):
    stats = {}
    for (s, r, t) in g.edge_types:
        if r in ['CALL','ARG2PARAM','RET2CALL','RET2LHS']:
            stats[r] = int(get_edge_index(g,(s,r,t)).size(1))
    if sum(stats.values()) == 0:
        print("⚠️  No inter-procedural edges found!")
    else:
        print("✓ Inter-procedural edges:", stats)
    return stats

def make_inter_mask(g: HeteroData):
    st = g[NODE_TYPE]; N = st.num_nodes
    inter_rel = {'CALL','ARG2PARAM','RET2CALL','RET2LHS',
                 'CALL_REV','ARG2PARAM_REV','RET2CALL_REV','RET2LHS_REV'}
    inter_nodes = set()
    for (s, r, t) in g.edge_types:
        if r in inter_rel:
            ei = get_edge_index(g, (s,r,t))
            if ei.numel() > 0:
                inter_nodes.update(ei[0].tolist()); inter_nodes.update(ei[1].tolist())
    device = (st.x_text.device if hasattr(st, 'x_text') else (st.x.device if hasattr(st, 'x') else 'cpu'))
    mask = torch.zeros(N, dtype=torch.bool, device=device)
    if inter_nodes: mask[list(inter_nodes)] = True
    return mask

# ------------------------------ dynamic shrinker ------------------------------
MAX_NODES_FOR_GPU  = 6000
MAX_EDGES_FOR_GPU  = 20000

def shrink_graph_if_needed(g: HeteroData) -> HeteroData:
    st = g[NODE_TYPE]; N  = st.num_nodes
    total_edges = 0
    for et in g.edge_types:
        total_edges += int(get_edge_index(g, et).size(1))
    if N <= MAX_NODES_FOR_GPU and total_edges <= MAX_EDGES_FOR_GPU:
        return g
    keep = set()
    inter_rel = {'CALL','ARG2PARAM','RET2CALL','RET2LHS',
                 'CALL_REV','ARG2PARAM_REV','RET2CALL_REV','RET2LHS_REV'}
    for (s,r,t) in g.edge_types:
        if r in inter_rel:
            ei = get_edge_index(g, (s,r,t))
            if ei.numel()>0:
                keep.update(ei[0].tolist()); keep.update(ei[1].tolist())
    random.seed(0)
    if len(keep) < min(N, MAX_NODES_FOR_GPU):
        others = [i for i in range(N) if i not in keep]
        random.shuffle(others)
        need = min(N, MAX_NODES_FOR_GPU) - len(keep)
        keep.update(others[:max(0, need)])
    keep = sorted(list(keep))
    idx_map = {old:i for i, old in enumerate(keep)}
    g2 = HeteroData()
    if hasattr(st, "x_text"):
        g2[NODE_TYPE].x_text = st.x_text[keep].to(dtype=torch.float16, device=st.x_text.device)
    if hasattr(st, "x"):
        g2[NODE_TYPE].x = st.x[keep].to(dtype=torch.float16, device=st.x.device)
    if hasattr(st, "nid"):
        g2[NODE_TYPE].nid = st.nid[keep]
    g2[NODE_TYPE].num_nodes = len(keep)
    for (s,r,t) in g.edge_types:
        ei = get_edge_index(g, (s,r,t))
        if ei.numel() == 0:
            g2[(s,r,t)].edge_index = ei; continue
        src, dst = ei
        keep_mask = [(u in idx_map and v in idx_map) for u,v in zip(src.tolist(), dst.tolist())]
        if not any(keep_mask):
            g2[(s,r,t)].edge_index = torch.zeros((2,0), dtype=torch.long, device=ei.device); continue
        src_f = [src[i].item() for i, m in enumerate(keep_mask) if m]
        dst_f = [dst[i].item() for i, m in enumerate(keep_mask) if m]
        src_r = torch.tensor([idx_map[u] for u in src_f], dtype=torch.long, device=ei.device)
        dst_r = torch.tensor([idx_map[v] for v in dst_f], dtype=torch.long, device=ei.device)
        if src_r.numel() > MAX_EDGES_FOR_GPU:
            perm = torch.randperm(src_r.numel(), device=ei.device)[:MAX_EDGES_FOR_GPU]
            src_r = src_r[perm]; dst_r = dst_r[perm]
        g2[(s,r,t)].edge_index = torch.stack([src_r, dst_r], dim=0)
    g2.__dict__["_aug_json_path"] = getattr(g, "_aug_json_path", None)
    return g2

# ------------------------------ model ------------------------------
import torch.utils.checkpoint as cp

class CAGatBlock(nn.Module):
    def __init__(self, in_ch, out_ch, heads, edge_types, dropout=0.1, use_skip=True):
        super().__init__()
        assert out_ch % heads == 0
        self.edge_types = edge_types
        self.use_skip   = use_skip
        self.convs = nn.ModuleDict({
            f"{s}_{r}_{t}": GATConv(in_ch, out_ch//heads, heads=heads, concat=True,
                                    add_self_loops=False, dropout=dropout)
            for (s,r,t) in edge_types
        })
        self.gates = nn.ParameterDict({ f"{s}_{r}_{t}": nn.Parameter(torch.zeros(1)) for (s,r,t) in edge_types })
        self.norm  = nn.LayerNorm(out_ch)
        self.drop  = nn.Dropout(dropout)
        self.proj  = nn.Linear(in_ch, out_ch) if in_ch != out_ch else nn.Identity()
        self.skip_alpha = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, g: HeteroData):
        out_dim = self.proj.out_features if isinstance(self.proj, nn.Linear) else x.size(1)
        out_accum = torch.zeros(x.size(0), out_dim, device=x.device, dtype=x.dtype)
        for (s,r,t) in self.edge_types:
            et = (s,r,t)
            ei = get_edge_index(g, et)
            if ei.numel()==0: continue
            key   = f"{s}_{r}_{t}"
            out = self.convs[key](x, ei)
            gate  = torch.sigmoid(self.gates[key])
            out_accum = out_accum + gate * out
        out_accum = self.drop(out_accum)
        res = self.proj(x)
        out_accum = self.skip_alpha * res + (1 - self.skip_alpha) * out_accum
        return self.norm(out_accum)

class CAGAT_ACC_InterProc(nn.Module):
    def __init__(self, in_dim, hidden, heads, layers, edge_types, dropout=0.1,
                 call_chain_attn=False, max_hops=3, sparse_threshold=5000):
        super().__init__()
        self.edge_types = edge_types
        self.hidden     = hidden
        self.projectors = nn.ModuleDict()
        self.blocks     = nn.ModuleList([
            CAGatBlock(hidden, hidden, heads, edge_types, dropout=dropout, use_skip=True)
            for _ in range(layers)
        ])
        self.intra_head = nn.Linear(hidden, hidden)
        he_init = nn.Linear(hidden, hidden); nn.init.xavier_uniform_(he_init.weight); nn.init.zeros_(he_init.bias)
        self.inter_head = he_init
        self.context_gru = nn.GRU(hidden, hidden, batch_first=True)
        self.lin_out   = nn.Linear(hidden, 1)
        self.call_chain_attn = call_chain_attn
        self.max_hops        = max_hops
        self.sparse_threshold= sparse_threshold
    def _project_in(self, x: torch.Tensor) -> torch.Tensor:
        d = x.size(1); key = f"proj_{d}"
        if key not in self.projectors:
            lin = nn.Linear(d, self.hidden, device=x.device, dtype=x.dtype)
            nn.init.kaiming_uniform_(lin.weight, a=math.sqrt(5))
            if lin.bias is not None:
                fan_in, _ = nn.init._calculate_fan_in_and_fan_out(lin.weight)
                bound = 1 / math.sqrt(fan_in); nn.init.uniform_(lin.bias, -bound, bound)
            self.projectors[key] = lin
        else:
            self.projectors[key] = self.projectors[key].to(device=x.device, dtype=x.dtype)
        return F.relu(self.projectors[key](x))
    def _call_chain_attn_sparse(self, call_ei, h, N):
        device = h.device
        idx = call_ei.to(device); vals = torch.ones(idx.size(1), device=device)
        A = torch.sparse_coo_tensor(idx, vals, size=(N, N))
        reach = A
        for _ in range(self.max_hops - 1):
            reach = reach + torch.sparse.mm(A, reach)
        neigh = reach.to_dense().clamp(max=1.0); neigh.fill_diagonal_(0)
        deg = neigh.sum(dim=1, keepdim=True).clamp(min=1)
        ctx = neigh @ h / deg
        return h + 0.1 * ctx
    def _call_chain_attn_bfs(self, call_ei, h, N):
        adj = defaultdict(list)
        for i in range(call_ei.size(1)):
            adj[call_ei[0,i].item()].append(call_ei[1,i].item())
        ctx = torch.zeros_like(h)
        for node_id in range(N):
            neighbors = []
            q = deque([(node_id,0)]); seen = {node_id}
            while q:
                cur, d = q.popleft()
                if d >= self.max_hops: continue
                for nb in adj.get(cur, []):
                    if nb not in seen:
                        seen.add(nb); neighbors.append(nb); q.append((nb, d+1))
            if neighbors: ctx[node_id] = h[neighbors].mean(dim=0)
        return h + 0.1 * ctx
    def forward(self, g: HeteroData):
        st = g[NODE_TYPE]; xs=[]
        if hasattr(st,"x_text"): xs.append(st.x_text)
        if hasattr(st,"x"):      xs.append(st.x.float())
        if len(xs) == 1: x = xs[0].to(device=xs[0].device, dtype=torch.float32)
        else: 
            target_device = xs[0].device; xs = [t.to(device=target_device, dtype=torch.float32) for t in xs]
            x = torch.cat(xs, dim=1)
        h = self._project_in(x)
        for blk in self.blocks:
            if USE_CHECKPOINT and h.requires_grad:
                h = cp.checkpoint(lambda inp: F.elu(blk(inp, g)), h, use_reentrant=False)
            else:
                h = F.elu(blk(h, g))
        inter_mask = make_inter_mask(g).to(h.device)
        h_mix = torch.where(inter_mask.unsqueeze(-1), self.inter_head(h), self.intra_head(h))
        et = (NODE_TYPE,'CALL',NODE_TYPE); ei = get_edge_index(g, et)
        if ei.numel() > 0:
            caller = h_mix[ei[0]]; callee = h_mix[ei[1]]
            seq = torch.stack([caller, callee], dim=1)  # [E,2,H]
            ctx,_ = self.context_gru(seq)
            callee_upd = ctx[:,1,:]
            deg = torch.zeros(h_mix.size(0), device=h_mix.device)
            deg.index_add_(0, ei[1], torch.ones(ei.size(1), device=h_mix.device))
            add = torch.zeros_like(h_mix); add.index_add_(0, ei[1], callee_upd)
            mask = (deg > 0).unsqueeze(-1)
            h_mix = torch.where(mask, h_mix + 0.2 * (add / deg.clamp(min=1).unsqueeze(-1)), h_mix)
        if self.call_chain_attn:
            call_ei = get_edge_index(g, (NODE_TYPE,'CALL',NODE_TYPE))
            if call_ei.numel() > 0:
                N = h_mix.size(0)
                h_mix = self._call_chain_attn_sparse(call_ei, h_mix, N) if N >= self.sparse_threshold else self._call_chain_attn_bfs(call_ei, h_mix, N)
        logit = self.lin_out(h_mix).squeeze(-1)
        return logit, h_mix

# ------------------------------ losses & metrics ------------------------------
def weighted_edge_smoothness(g: HeteroData, node_logit: torch.Tensor):
    loss = 0.0; count = 0
    for (s,r,t) in g.edge_types:
        w = EDGE_WEIGHTS.get(r, 0.0); 
        if w == 0.0: continue
        ei = get_edge_index(g, (s,r,t)); 
        if ei.numel()==0: continue
        u, v = ei[0], ei[1]
        loss = loss + w * F.l1_loss(node_logit[u], node_logit[v])
        count += 1
    return loss / max(1, count)

def inter_procedural_contrastive_loss(g: HeteroData, emb: torch.Tensor, temperature=0.1):
    loss = 0.0; count = 0
    for r in ['CALL','ARG2PARAM','RET2CALL']:
        et = (NODE_TYPE, r, NODE_TYPE)
        ei = get_edge_index(g, et)
        if ei.numel()==0: continue
        a = emb[ei[0]]; b = emb[ei[1]]
        pos_sim = F.cosine_similarity(a, b, dim=-1)
        loss = loss - torch.log(torch.sigmoid(pos_sim / temperature)).mean()
        count += 1
    return loss / max(1, count)

def path_supervision_loss(g: HeteroData, node_logit: torch.Tensor, paths_node_ids: list):
    if not paths_node_ids: 
        return node_logit.new_zeros(())
    st = g[NODE_TYPE]; id2pos = None
    if hasattr(st, "nid"):
        ids = st.nid.view(-1).tolist(); id2pos = {nid:i for i,nid in enumerate(ids)}
    loss = 0.0; cnt  = 0
    for path in paths_node_ids:
        idxs = []
        for nid in path:
            j = id2pos.get(_coerce_int(nid)) if id2pos is not None else (nid if (isinstance(nid, int) and 0 <= nid < st.num_nodes) else None)
            if j is None: idxs=[]; break
            idxs.append(j)
        if len(idxs) < 2: continue
        pl = node_logit[idxs]
        # enforce increasing scores root..sink
        loss = loss + sum(F.relu(pl[i] - pl[i+1]) for i in range(len(pl)-1)) / (len(pl)-1)
        cnt += 1
    return loss / max(1, cnt) if cnt>0 else node_logit.new_zeros(())

def pred_threshold(epoch, total_epochs):
    return 0.30 + 0.20 * (epoch / max(1, total_epochs))

def compute_inter_or_fallback_metrics(g, node_logit, y, thr=0.5):
    mask = make_inter_mask(g)
    if y.sum() == 0:
        return {"note": "no_positives"}
    if mask.sum() > 0 and (y[mask].sum() > 0):
        inter_logits = node_logit[mask]; inter_labels = y[mask]
        pred = (torch.sigmoid(inter_logits) > thr).float()
        tp = (pred * inter_labels).sum()
        prec = tp / (pred.sum() + 1e-8); rec  = tp / (inter_labels.sum() + 1e-8)
        acc  = (pred == inter_labels).float().mean()
        return {'inter_acc': float(acc), 'inter_precision': float(prec), 'inter_recall': float(rec)}
    pred = (torch.sigmoid(node_logit) > thr).float()
    tp = (pred * y).sum()
    prec = tp / (pred.sum() + 1e-8); rec  = tp / (y.sum() + 1e-8)
    acc  = (pred == y).float().mean()
    return {'inter_acc': float(acc), 'inter_precision': float(prec), 'inter_recall': float(rec), 'note': 'fallback_global'}

def get_loss_weights(epoch, total_epochs):
    prog = epoch / total_epochs
    return {
        'node':     LAMBDA_NODE_BCE_BASE,
        'smooth':   LAMBDA_EDGE_SMOOTH_BASE    + 0.15 * prog,  # 0.05 → 0.20
        'contrast': LAMBDA_INTER_CONTRAST_BASE + 0.15 * prog,  # 0.05 → 0.20
        'path':     LAMBDA_PATH_SUPER_BASE     + 0.20 * prog,  # 0.10 → 0.30
    }

# ------------------------------ Weak-supervision miner (fast + robust static seeds) ------------------------------
FAST_MINER          = True        # if False, also use regex over JSON node text
TIME_BUDGET_SEC     = 2.0         # per-graph budget
MAX_JSON_BYTES      = 2_000_000
MAX_FRONTIER        = 4096
MAX_VISITED         = 15000
MINER_MAX_HOPS      = 5
MINER_LIMIT_PATHS   = 8

# Static patterns kept (more robust, broader coverage). These are *additive* with structural seeds.
SINK_NAME_PATTERNS = [
    r"\b(exec|system|popen|spawn|CreateProcess|ShellExecute|eval|Runtime\.exec|ProcessBuilder)\b",
    r"\b(sql|execute(Query|Update)|Statement\.execute|rawQuery|PreparedStatement)\b",
    r"\b(send|write|print|fprintf|fwrite|OutputStream\.write|FileWriter|FileOutputStream)\b",
    r"\b(deserialize|ObjectInputStream|pickle\.loads|JSON\.parse)\b",
]
ROOT_NAME_PATTERNS = [
    r"\b(argv|stdin|getenv|getopt|request|req|input|params?|query|body|post|recv|read|Scanner|BufferedReader)\b",
    r"\b(getParameter|getHeader|getQueryString|getInputStream|readLine)\b",
]

_JSON_CACHE = {}; _JSON_KEYS  = []; _JSON_CACHE_LIMIT = 512
def _json_cache_get(path):
    if not path or not os.path.exists(path): return None
    try:
        if os.path.getsize(path) > MAX_JSON_BYTES: return None
    except Exception: pass
    if path in _JSON_CACHE: return _JSON_CACHE[path]
    j = load_aug_json(path)
    if j is not None:
        _JSON_CACHE[path] = j; _JSON_KEYS.append(path)
        if len(_JSON_KEYS) > _JSON_CACHE_LIMIT:
            old = _JSON_KEYS.pop(0); _JSON_CACHE.pop(old, None)
    return j

def _load_nodes_map_fast(json_path:str):
    j = _json_cache_get(json_path)
    if not isinstance(j, dict) or not isinstance(j.get("nodes"), list):
        return [], {}
    nodes = j["nodes"]; id2node = {}
    for n in nodes:
        if isinstance(n, dict) and "_id" in n:
            id2node[_coerce_int(n["_id"])] = n
    return nodes, id2node

def _node_text_from_json(n:dict) -> str:
    for k in ("code","name","methodFullName","signature","call","label","nodeType"):
        v = n.get(k)
        if isinstance(v,str) and v.strip(): return v
    return ""

def _regex_any(patterns, s):
    if not isinstance(s, str): return False
    for p in patterns:
        if re.search(p, s, flags=re.I): return True
    return False

def _build_adj_fast(g: HeteroData):
    allowed_rels = {'CALL','ARG2PARAM','RET2CALL','RET2LHS','DFG'}
    adj = defaultdict(list)
    for (s,r,t) in g.edge_types:
        if r not in allowed_rels: continue
        ei = get_edge_index(g,(s,r,t))
        if ei.numel()==0: continue
        src, dst = ei
        buckets = defaultdict(list)
        for u,v in zip(src.tolist(), dst.tolist()): buckets[u].append(v)
        for u, vs in buckets.items():
            adj[u].extend(vs[:64])  # cap out-degree
    return adj

def _bfs_paths_bounded(adj, starts, targets, max_hops=5, limit=8, time_budget=1.0):
    t0 = time.monotonic(); paths = []; targets = set(targets)
    visited_budget = 0
    for s in starts:
        q = deque([(s, [s])])
        while q and len(paths) < limit:
            if (time.monotonic() - t0) > time_budget: return paths
            cur, path = q.popleft()
            if len(path) - 1 > max_hops: continue
            if cur in targets and len(path) > 1:
                paths.append(path[:])
                if len(paths) >= limit: break
            nb_list = adj.get(cur, [])[:64]
            for nb in nb_list:
                if nb in path: continue
                q.append((nb, path + [nb])); visited_budget += 1
                if visited_budget > MAX_VISITED: return paths
            if len(q) > MAX_FRONTIER:
                while len(q) > MAX_FRONTIER: q.pop()
    return paths

def mine_roots_sinks_and_paths(g: HeteroData, json_path: str,
                               max_hops=MINER_MAX_HOPS, limit_paths=MINER_LIMIT_PATHS):
    t0 = time.monotonic()
    roots, sinks, paths = [], [], []
    # (A) structural seeds (always available; robust)
    # sinks = CALL callees + RET2CALL/RET2LHS targets
    for r in ('RET2LHS','RET2CALL','CALL'):
        ei = get_edge_index(g,(NODE_TYPE,r,NODE_TYPE))
        if ei.numel()>0: sinks.extend(ei[1].unique().tolist())
    sinks = list(set(sinks))[:256]
    # roots = ARG2PARAM sources + high-outdegree DFG sources
    ei = get_edge_index(g,(NODE_TYPE,'ARG2PARAM',NODE_TYPE))
    if ei.numel()>0: roots = ei[0].unique().tolist()[:256]
    dfg = get_edge_index(g,(NODE_TYPE,'DFG',NODE_TYPE))
    if dfg.numel()>0:
        deg = torch.zeros(g[NODE_TYPE].num_nodes, device=dfg.device); deg.index_add_(0, dfg[0], torch.ones(dfg.size(1), device=deg.device))
        high = torch.topk(deg, k=min(128, deg.numel())).indices.tolist()
        roots = list(set((roots or []) + high))
    # (B) optional lexical seeds from JSON node text (if enabled)
    if (not FAST_MINER) and json_path:
        nodes, id2node = _load_nodes_map_fast(json_path)
        if hasattr(g[NODE_TYPE], "nid") and id2node:
            g_ids = g[NODE_TYPE].nid.view(-1).tolist()
            hit_roots=set(); hit_sinks=set()
            for nid in g_ids[:4096]:
                n = id2node.get(_coerce_int(nid)); 
                if not isinstance(n, dict): continue
                txt = _node_text_from_json(n)
                if _regex_any(SINK_NAME_PATTERNS, txt): hit_sinks.add(nid)
                if _regex_any(ROOT_NAME_PATTERNS, txt): hit_roots.add(nid)
            if hit_sinks: sinks = list(set(sinks + list(hit_sinks)))
            if hit_roots: roots = list(set(roots + list(hit_roots)))
    # Mine paths between roots and sinks
    if roots and sinks:
        adj = _build_adj_fast(g)
        rem = max(0.2, TIME_BUDGET_SEC - (time.monotonic() - t0))
        paths = _bfs_paths_bounded(adj, starts=roots, targets=sinks,
                                   max_hops=max_hops, limit=limit_paths, time_budget=rem)
    # fallback: stub edges
    if not paths and roots and sinks:
        for r in roots[:4]:
            for s in sinks[:4]:
                if r != s:
                    paths.append([r, s])
                    if len(paths) >= limit_paths: break
            if len(paths) >= limit_paths: break
    return roots, sinks, paths

def _ensure_inter_pos(g, y_soft, paths):
    """Guarantee at least one positive inside the inter-procedural slice."""
    mask = make_inter_mask(g)
    if (y_soft[mask] > 0.5).any(): 
        return y_soft, paths
    # try to promote a node on a mined path that lies in the slice
    promoted = False
    id2pos = None
    if hasattr(g[NODE_TYPE],"nid"):
        ids = g[NODE_TYPE].nid.view(-1).tolist(); id2pos = {nid:i for i,nid in enumerate(ids)}
    for path in paths:
        for nid in path[::-1]:
            j = id2pos.get(_coerce_int(nid)) if id2pos is not None else (nid if isinstance(nid,int) else None)
            if j is not None and j < y_soft.numel() and mask[j]:
                y_soft[j] = max(y_soft[j], 0.9); promoted = True; break
        if promoted: break
    # if still none, pick high-degree CALL callee
    if not promoted:
        ei = get_edge_index(g,(NODE_TYPE,'CALL',NODE_TYPE))
        if ei.numel()>0:
            deg = torch.zeros(y_soft.numel(), device=y_soft.device)
            deg.index_add_(0, ei[1], torch.ones(ei.size(1), device=y_soft.device))
            j = int(deg.argmax().item())
            if mask[j]: y_soft[j] = max(y_soft[j], 0.9)
    return y_soft, paths

def build_labels_with_weak_supervision(g: HeteroData):
    json_path = getattr(g, "_aug_json_path", None)
    # 1) try hard labels
    y = node_sink_labels_from_json(json_path, g); hard_paths = read_paths_from_json(json_path)
    if y is not None and (float(y.sum().item()) > 0 or (hard_paths and len(hard_paths)>0)):
        return y.to(y.device), hard_paths, {"mode":"hard_json"}
    # 2) mine paths
    roots, sinks, mined_paths = mine_roots_sinks_and_paths(g, json_path,
                                                           max_hops=MINER_MAX_HOPS,
                                                           limit_paths=MINER_LIMIT_PATHS)
    st = g[NODE_TYPE]
    device = (st.x_text.device if hasattr(st,"x_text") else (st.x.device if hasattr(st,"x") else "cpu"))
    N = st.num_nodes
    y_soft = torch.zeros(N, dtype=torch.float32, device=device)
    # soft labels along paths: ramp 0.3 → 0.9
    if hasattr(st,"nid"):
        ids = st.nid.view(-1).tolist(); id2pos = {nid:i for i,nid in enumerate(ids)}
        for path in mined_paths:
            L = max(2, len(path)); 
            for k, nid in enumerate(path):
                j = id2pos.get(_coerce_int(nid)); 
                if j is None or j >= N: continue
                tgt = 0.3 + 0.6 * (k / (L-1))  # root ~0.3 … sink ~0.9
                y_soft[j] = max(y_soft[j], tgt)
    # ensure at least one inter-proc positive
    y_soft, mined_paths = _ensure_inter_pos(g, y_soft, mined_paths)
    return y_soft, mined_paths, {"mode":"weak_mined", "roots": len(roots), "sinks": len(sinks), "paths": len(mined_paths)}

print(f"GC_DIR={GC_DIR}")
print(f"UNIFIED_JSON_DIRS={UNIFIED_JSON_DIRS}")
print(f"AUG_JSON_DIRS={AUG_JSON_DIRS}")

# ------------------------------ training ------------------------------
def train(
    gc_dir=GC_DIR,
    logdir=LOGDIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    hidden=HIDDEN,
    heads=HEADS,
    layers=LAYERS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    save_every=SAVE_EVERY_STEPS,
    warmup_max_steps=None,    # if set, run bounded warm-up
):
    ds = GraphDir(gc_dir)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=lambda xs: xs[0], pin_memory=False)

    sample = safe_load(ds.paths[0], map_location="cpu")
    st = sample[NODE_TYPE]
    in_dim = (st.x_text.size(1) if hasattr(st, "x_text") else 0) + (st.x.size(1) if hasattr(st, "x") else 0)
    etypes = tuple(sample.edge_types)

    if hidden % heads != 0:
        new_hidden = (hidden // heads + 1) * heads
        print(f"[CFG] HIDDEN={hidden} not divisible by HEADS={heads} → adjusting to {new_hidden}")
        hidden = new_hidden

    model = CAGAT_ACC_InterProc(
        in_dim=in_dim, hidden=hidden, heads=heads, layers=layers,
        edge_types=etypes, dropout=DROPOUT,
        call_chain_attn=CALL_CHAIN_ATTN, max_hops=CALL_CHAIN_MAX_HOPS,
        sparse_threshold=CALL_CHAIN_SPARSE_NTHRESH
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    os.makedirs(logdir, exist_ok=True)
    ckpt_path       = os.path.join(logdir, "ckpt_cagat_acc_interproc.pt")
    best_path       = os.path.join(logdir, "best_model.pt")
    summary_path    = os.path.join(logdir, "train_summary.json")
    log_jsonl_path  = os.path.join(logdir, "training_log.jsonl")

    best_inter_recall = 0.0
    log_jsonl_fh = open(log_jsonl_path, "a", encoding="utf-8")

    def save(tag="SAVE", steps=0, loss_avg=0.0, extras=None, best=False):
        payload = {"model": model.state_dict()}
        torch.save(payload, best_path if best else ckpt_path)
        out = {"loss_avg": loss_avg, "steps": steps, "epochs": epochs}
        if isinstance(extras, dict): out.update(extras)
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(out, f, indent=2)
        tqdm.write(f"[{tag}] -> {(best_path if best else ckpt_path)}")

    atexit.register(lambda: save("ATEXIT", steps=0, loss_avg=0.0))
    atexit.register(lambda: log_jsonl_fh.close())

    steps, loss_sum = 0, 0.0
    graphs_seen = graphs_with_pos = graphs_with_inter_pos = 0

    total_steps = (warmup_max_steps if warmup_max_steps is not None else epochs * len(dl))
    print(f"[TRAIN] Device: {DEVICE}")
    print(f"[TRAIN] Sample: N={st.num_nodes}, in_dim={in_dim}, edge_types={len(etypes)}")
    validate_interprocedural_coverage(sample)

    phase = "Warmup" if warmup_max_steps is not None else "Training"
    pbar = tqdm(total=total_steps, desc=f"{phase}", unit="step", dynamic_ncols=True)

    try:
        epoch_iter = [1] if warmup_max_steps is not None else range(1, epochs + 1)
        for ep in epoch_iter:
            w = get_loss_weights(ep, epochs if warmup_max_steps is None else 1)
            for i, g in enumerate(dl, 1):
                if warmup_max_steps is not None and steps >= warmup_max_steps:
                    break
                tried_shrink = False
                while True:
                    try:
                        g_work = g.to(DEVICE, non_blocking=False)
                        with autocast_ctx(enabled=USE_AMP):
                            node_logit, h = model(g_work)
                            st_g = g_work[NODE_TYPE]

                            # Hard labels → else inter-aware weak labels
                            y = node_sink_labels_from_json(getattr(g_work, "_aug_json_path", None), g_work)
                            paths = read_paths_from_json(getattr(g_work, "_aug_json_path", None))
                            label_info = None
                            if (y is None) or (float(y.sum().item()) == 0 and not paths):
                                y, paths, label_info = build_labels_with_weak_supervision(g_work)

                            y = y.to(node_logit.device)
                            inter_mask = make_inter_mask(g_work)

                            graphs_seen += 1
                            pos_all   = int((y > 0.5).sum().item())
                            pos_inter = int(((y > 0.5).float() * inter_mask.float()).sum().item())
                            if pos_all > 0: graphs_with_pos += 1
                            if pos_inter > 0: graphs_with_inter_pos += 1

                            # BCE supports soft targets
                            pos = (y > 0.5).sum()
                            neg = y.numel() - pos
                            pos_weight = (neg / (pos + 1e-6)).clamp_(1.0, 100.0)
                            node_loss = F.binary_cross_entropy_with_logits(node_logit, y, pos_weight=pos_weight)

                            # Aux losses
                            smooth    = weighted_edge_smoothness(g_work, node_logit)
                            contra    = inter_procedural_contrastive_loss(g_work, h)
                            path_loss = path_supervision_loss(g_work, node_logit, paths) if paths else node_logit.new_zeros(())

                            loss = (w['node'] * node_loss + w['smooth'] * smooth +
                                    w['contrast'] * contra + w['path'] * path_loss)

                        # step
                        opt.zero_grad(set_to_none=True)
                        SCALER.scale(loss).backward()
                        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                        SCALER.step(opt); SCALER.update()

                        # metrics
                        thr = pred_threshold(ep, epochs if warmup_max_steps is None else 1)
                        metrics = compute_inter_or_fallback_metrics(g_work, node_logit.detach(), (y > 0.5).float(), thr=thr)

                        steps += 1; loss_sum += float(loss.detach().cpu())
                        postfix = dict(ep=ep, step=steps, loss=f"{loss_sum/steps:.4f}",
                                       i_acc=(f"{metrics.get('inter_acc', 0):.3f}" if metrics else "N/A"),
                                       i_rec=(f"{metrics.get('inter_recall', 0):.3f}" if metrics else "N/A"))
                        if "note" in metrics: postfix["note"] = metrics["note"]
                        pbar.set_postfix(**postfix); pbar.update(1)

                        # jsonl
                        if (steps % LOG_JSONL_EVERY) == 0:
                            log_line = {
                                "epoch": ep, "step": steps,
                                "loss": float(loss.detach().cpu()),
                                "node_loss": float(node_loss.detach().cpu()),
                                "smooth": float(smooth.detach().cpu()) if torch.is_tensor(smooth) else float(smooth),
                                "contrast": float(contra.detach().cpu()) if torch.is_tensor(contra) else float(contra),
                                "path_loss": float(path_loss.detach().cpu()) if torch.is_tensor(path_loss) else float(path_loss),
                                "pos_any": pos_all, "pos_inter": pos_inter,
                                **{k: float(v) for k,v in metrics.items() if isinstance(v, (int,float))}
                            }
                            if isinstance(metrics.get("note"), str): log_line["note"] = metrics["note"]
                            if label_info: log_line["label_info"] = label_info
                            log_jsonl_fh.write(json.dumps(log_line) + "\n"); log_jsonl_fh.flush()

                        inter_rec = float(metrics.get('inter_recall', 0.0))
                        if inter_rec > best_inter_recall and warmup_max_steps is None:
                            best_inter_recall = inter_rec
                            save("BEST (inter_recall↑)", steps=steps, loss_avg=loss_sum/steps, extras=metrics, best=True)

                        # tidy
                        del node_logit, h, y, loss, node_loss, smooth, contra, path_loss, g_work
                        if (i % CLEAR_CACHE_EVERY) == 0 and DEVICE.type == "cuda":
                            torch.cuda.empty_cache()
                        break

                    except RuntimeError as e:
                        if 'out of memory' in str(e).lower():
                            torch.cuda.empty_cache()
                            if not tried_shrink:
                                g = shrink_graph_if_needed(g); tried_shrink = True
                                tqdm.write("[INFO] Retrying with shrunken graph to avoid OOM.")
                                continue
                            elif SKIP_ON_OOM:
                                tqdm.write("[WARN] Skipping one graph due to CUDA OOM even after shrink."); break
                            else:
                                raise
                        else:
                            raise

            if warmup_max_steps is not None and steps >= warmup_max_steps:
                break

    except Exception:
        save("FINAL (EXC)", steps=steps, loss_avg=(loss_sum / max(1, steps)))
        traceback.print_exc(); raise
    finally:
        save("FINAL", steps=steps, loss_avg=(loss_sum / max(1, steps)))
        pbar.close(); log_jsonl_fh.close()

    # ----------- training report -----------
    report = {
        "phase": ("warmup" if warmup_max_steps is not None else "train"),
        "steps": steps,
        "avg_loss": loss_sum/max(1,steps),
        "graphs_seen": graphs_seen,
        "graphs_with_any_pos": graphs_with_pos,
        "graphs_with_inter_pos": graphs_with_inter_pos,
        "best_inter_recall": (best_inter_recall if warmup_max_steps is None else None),
        "log_jsonl": log_jsonl_path,
        "summary_json": summary_path,
        "ckpt_last": os.path.join(logdir, "ckpt_cagat_acc_interproc.pt"),
        "ckpt_best": (os.path.join(logdir, "best_model.pt") if warmup_max_steps is None else None),
    }
    # Persist report
    with open(os.path.join(logdir, "training_report.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    if warmup_max_steps is not None:
        print(f"[WARMUP] summary: {report}")
    else:
        print(f"[TRAIN] summary: {report}")

# ------------------------------ Convenience wrappers ------------------------------
def warmup_check(max_steps=300):
    print("[WARMUP] starting…")
    train(warmup_max_steps=max_steps)

def full_run():
    print("[FULL] starting…")
    train()

# ========================== toggles (set & run cell) ==========================
DO_WARMUP   = True    # quick sanity
DO_FULL_RUN = False   # full training later

# Miner knobs (you can tweak live)
FAST_MINER      = True     # False => also use regex on JSON node text
TIME_BUDGET_SEC = 2.0      # try 3.0–5.0 if your GPU/CPU can handle

# Make log dir
os.makedirs(LOGDIR, exist_ok=True)

# Execute
if DO_WARMUP:
    warmup_check(max_steps=300)
if DO_FULL_RUN:
    full_run()


[AMP] torch=2.4.1+cu121 device=cuda
GC_DIR=Dataset/train/hetero_ready_gcbert
UNIFIED_JSON_DIRS=['Dataset/train/unified_json', 'Dataset/train/unified']
AUG_JSON_DIRS=['Dataset/train/unified_aug', 'Dataset/train/unified']
[WARMUP] starting…
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert
[TRAIN] Device: cuda
[TRAIN] Sample: N=3141, in_dim=793, edge_types=14
✓ Inter-procedural edges: {'CALL': 608, 'ARG2PARAM': 1097, 'RET2CALL': 608, 'RET2LHS': 96}


Warmup:   0%|          | 0/300 [00:00<?, ?step/s]c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\.venv\Lib\site-packages\torch\utils\checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
Warmup:   1%|          | 2/300 [00:01<02:32,  1.96step/s, ep=1, i_acc=0.541, i_rec=0.000, loss=0.8874, step=2]


[FINAL] -> out/cagat_acc_interproc_train\ckpt_cagat_acc_interproc.pt


KeyboardInterrupt: 